# 📊 Évaluation et Visualisation des Résultats

## Objectifs
1. Charger les modèles entraînés
2. Évaluer les performances sur de nouvelles données
3. Analyser les erreurs
4. Visualiser les résultats
5. Générer un rapport final

In [ ]:
# Importations
import sys
import os
sys.path.append('..')

from src.utils.config import Config
from src.data.loader import DataLoader
from src.models.binary_relevance import BinaryRelevanceModel
from src.models.classifier_chains import ClassifierChainsModel
from src.utils.metrics import MultiLabelEvaluator
from src.utils.visualization import ResultsVisualizer

import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
from pathlib import Path

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Initialisation
print("🚀 Initialisation du notebook d'évaluation")

# Charger la configuration
config = Config()
data_config = config.get_data_config()

# Initialiser les composants
loader = DataLoader()
evaluator = MultiLabelEvaluator(loader.spark)
visualizer = ResultsVisualizer()

print("✅ Composants initialisés")

## 📥 Chargement des données de test

In [ ]:
# Charger les données de test
test_data_path = os.path.join(data_config['processed_path'], "test_data.parquet")

# Si le fichier de test spécifique n'existe pas, utiliser un échantillon
if not os.path.exists(test_data_path):
    print(f"⚠️  Fichier de test non trouvé: {test_data_path}")
    print("📥 Chargement d'un échantillon depuis les données complètes...")
    
    # Charger les données complètes
    all_data_path = os.path.join(data_config['processed_path'], "data_with_features.parquet")
    if os.path.exists(all_data_path):
        all_data = loader.spark.read.parquet(all_data_path)
        
        # Créer un échantillon de test
        test_data = all_data.sample(fraction=0.1, seed=42)
        test_data_path = os.path.join(data_config['processed_path'], "test_sample.parquet")
        loader.save_data(test_data, test_data_path, format='parquet')
        print(f"✅ Échantillon de test créé: {test_data_path}")
    else:
        print(" Aucune donnée disponible")
        exit()
else:
    print(f" Chargement des données de test: {test_data_path}")

# Charger les données
test_df = loader.spark.read.parquet(test_data_path)
print(f" Données de test chargées: {test_df.count():,} articles")

# Vérifier les colonnes
required_cols = ['id', 'categories', 'features']
missing_cols = [col for col in required_cols if col not in test_df.columns]

if missing_cols:
    print(f" Colonnes manquantes: {missing_cols}")
else:
    print(" Toutes les colonnes nécessaires sont présentes")